# 05.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/06-hyperparameter-tuning/solutions.ipynb)

## Exercise 1

a) 3 × 2 × 4 = **24** settings.
b) 24 × 5 = 120 CV fits, + 1 refit = **121** fits.
c) 121 × 0.3 s ≈ **36 s** on one core; about **4.5 s** on 8 cores (searches parallelise almost perfectly because the fits are independent).

In [1]:
import os, logging
os.environ["PYTHONWARNINGS"] = "ignore"
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
auc_logreg = roc_auc_score(y_test, logreg.predict_proba(X_test)[:, 1])
print(f"Logistic regression baseline, test AUC: {auc_logreg:.3f}")

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # the same 5 folds for every setting

3,317 customers, churn rate 43.6%, train 2,487 / test 830
Logistic regression baseline, test AUC: 0.741


## Exercise 2

In [2]:
import time
from sklearn.model_selection import GridSearchCV
grid3 = {"learning_rate": [0.01, 0.03, 0.1], "num_leaves": [4, 8, 31], "min_child_samples": [10, 40, 80]}
t = time.time()
g3 = GridSearchCV(lgb.LGBMClassifier(n_estimators=200, verbose=-1, random_state=42, n_jobs=1),
                  grid3, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print(f"{3*3*3} settings × 5 folds = {27*5} fits (+1 refit) in {time.time()-t:.1f} s")
print("best:", g3.best_params_, f"CV AUC {g3.best_score_:.3f}")

27 settings × 5 folds = 135 fits (+1 refit) in 1.8 s
best: {'learning_rate': 0.01, 'min_child_samples': 40, 'num_leaves': 8} CV AUC 0.744


Three times the fits, and the best CV AUC moves by a few thousandths at most, far less than the fold std. The extra dimension cost time and bought nothing measurable.

## Exercise 3

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
rf_search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=200, random_state=42),
    {"max_depth": randint(3, 16), "min_samples_leaf": randint(1, 51), "max_features": ["sqrt", 0.5, 1.0]},
    n_iter=15, cv=cv, scoring="roc_auc", random_state=42, n_jobs=-1).fit(X_train, y_train)
print("best:", rf_search.best_params_, f"CV AUC {rf_search.best_score_:.3f}")

best: {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 33} CV AUC 0.745


The tuned forest lands in the same 0.74 band as tuned LightGBM and logistic regression. On this table, model choice and tuning both matter far less than the features themselves.

## Exercise 4

In [4]:
import optuna

def objective(trial):
    model = lgb.LGBMClassifier(
        n_estimators=200, verbose=-1, random_state=42, n_jobs=1,
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 2, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100))
    return cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()

for seed in [0, 1, 2]:
    s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    s.optimize(objective, n_trials=15)
    p = s.best_params
    print(f"seed {seed}: lr={p['learning_rate']:.3f} leaves={p['num_leaves']:>2} "
          f"min_child={p['min_child_samples']:>2}  CV AUC {s.best_value:.3f}")

seed 0: lr=0.006 leaves= 5 min_child=91  CV AUC 0.742


seed 1: lr=0.025 leaves= 4 min_child=25  CV AUC 0.743


seed 2: lr=0.032 leaves= 2 min_child=81  CV AUC 0.742


The best *scores* agree to within about 0.005; the best *settings* can look quite different. That means the score surface is flat near the top: many settings are equally good. Do not read meaning into the exact numbers Optuna picks.